# Curva de geração de vapor × consumo de gás — caldeiras 208, 502 e 550
- **Eixo X:** consumo de gás (Nm³/h) · **Eixo Y:** geração de vapor (t/h) · base horária
- Só horas **em operação** (gás e vapor acima de 5% do p95), após limpeza: negativos → 0 e lacunas curtas de compressão preenchidas (até 12 h)
- Ajuste **linear** (vapor = a + b·gás) e **quadrático** por caldeira e por ano, com **uma tabela por ano**

In [0]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BRONZE_DIR = next(p for p in [os.path.abspath(os.path.join(os.getcwd(), '..', '1_bronze', 'processo_dados')), '/Workspace/Previsao de consumo de vapor/1_bronze/processo_dados'] if os.path.isdir(p))
RESULT_DIR = os.path.join(os.path.dirname(os.path.dirname(BRONZE_DIR)), '3_gold'); print(f'Bronze: {BRONZE_DIR}')
CALDEIRAS = {'208': ('11FI208-02', '11FC208-106'), '502': ('11FI502-30', '11FC502-01'), '550': ('21FI550-30', '21FC550-72')}  # (vapor t/h, gás Nm³/h)
LIMITE_FFILL_H = 12  # repete o último valor por até 12 h (compressão do historiador)
FRAC_OPERACAO = 0.05  # hora em operação: gás e vapor > 5% do p95 da própria tag
MIN_HORAS_ANO = 100  # mínimo de horas em operação para ajustar a curva do ano
COR_ANO = {2023: '#2a78d6', 2024: '#eb6834', 2025: '#1baf7a', 2026: '#eda100'}  # cor fixa por ano

norm = lambda t: str(t).upper().replace('.PV', '').strip()
arquivos = {}
for root, _, files in os.walk(BRONZE_DIR):
    for f in files:
        if f.endswith('.csv') and not re.search(r'\d{4}-\d{2}-\d{2}', f):
            arquivos.setdefault(norm(f.rsplit('.csv', 1)[0]), []).append(os.path.join(root, f))

def carregar(tag):
    frames = [pd.read_csv(f) for f in sorted(arquivos.get(norm(tag), []))]
    if not frames:
        raise FileNotFoundError(f"Tag {tag} não encontrada no bronze")
    df = pd.concat(frames)
    df['ts'] = pd.to_datetime(df['ts'], errors='coerce')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    s = df.dropna(subset=['ts']).drop_duplicates('ts').set_index('ts')['value'].sort_index()
    return s.clip(lower=0).resample('1h').mean().ffill(limit=LIMITE_FFILL_H)

dados = {}
for c, (tag_v, tag_g) in CALDEIRAS.items():
    df = pd.DataFrame({'vapor': carregar(tag_v), 'gas': carregar(tag_g)}).dropna()
    lim_v, lim_g = FRAC_OPERACAO * df['vapor'].quantile(0.95), FRAC_OPERACAO * df['gas'].quantile(0.95)
    op = df[(df['vapor'] > lim_v) & (df['gas'] > lim_g)].copy()
    op['ano'] = op.index.year
    dados[c] = op
    print(f"Caldeira {c}: {len(df):,} h com vapor e gás | {len(op):,} h em operação | anos: {sorted(op['ano'].unique().tolist())}")

In [0]:
linhas, curvas = [], {}
for c, op in dados.items():
    for ano, g in op.groupby('ano'):
        if len(g) < MIN_HORAS_ANO:
            continue
        x, y = g['gas'].values, g['vapor'].values
        b, a = np.polyfit(x, y, 1)
        q2, q1, q0 = np.polyfit(x, y, 2)
        r2_lin = 1 - ((y - (a + b * x)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        r2_quad = 1 - ((y - (q0 + q1 * x + q2 * x ** 2)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
        curvas[(c, ano)] = (a, b, q0, q1, q2, x.min(), x.max())
        linhas.append({'ano': ano, 'caldeira': c, 'horas_operacao': len(g), 'gas_min (Nm3/h)': x.min(), 'gas_max (Nm3/h)': x.max(), 'vapor_medio (t/h)': y.mean(), 'a (t/h)': a, 'b (t/Nm3)': b, 'R2_linear': r2_lin, 'q0': q0, 'q1': q1, 'q2': q2, 'R2_quadratico': r2_quad, 'vapor_por_1000Nm3 (t)': np.median(y / x) * 1000, 'gas_por_t_vapor (Nm3/t)': np.median(x / y)})
tab = pd.DataFrame(linhas).sort_values(['ano', 'caldeira']).reset_index(drop=True)
for ano, t in tab.groupby('ano'):
    print(f"=== {ano} — curva vapor × gás por caldeira ===")
    display(t.drop(columns='ano').round(6))
os.makedirs(RESULT_DIR, exist_ok=True)
tab.to_csv(f"{RESULT_DIR}/curva_gas_vapor_por_ano.csv", index=False)
print(f"Tabela salva em {RESULT_DIR}/curva_gas_vapor_por_ano.csv")

In [0]:
for c, op in dados.items():
    anos_c = sorted(a for (cc, a) in curvas if cc == c)
    if not anos_c:
        continue
    fig, axes = plt.subplots(1, len(anos_c) + 1, figsize=(5.2 * (len(anos_c) + 1), 4.6), sharex=True, sharey=True)
    for ax, ano in zip(axes, anos_c):
        g = op[op['ano'] == ano].sample(min(4000, int((op['ano'] == ano).sum())), random_state=42)
        a, b, q0, q1, q2, xmin, xmax = curvas[(c, ano)]
        xx = np.linspace(xmin, xmax, 100)
        ax.scatter(g['gas'], g['vapor'], s=6, alpha=0.25, color=COR_ANO.get(ano, '#2a78d6'), linewidths=0)
        r2 = tab.loc[(tab['caldeira'] == c) & (tab['ano'] == ano), 'R2_linear'].iloc[0]
        ax.set_title(f"{c} — {ano}  (R² linear = {r2:.3f})", fontsize=10)
        ax.set_xlabel('Consumo de gás (Nm³/h)')
        ax.grid(alpha=0.3)
    axes[0].set_ylabel('Geração de vapor (t/h)')
    for ano in anos_c:
        a, b, q0, q1, q2, xmin, xmax = curvas[(c, ano)]
        xx = np.linspace(xmin, xmax, 100)
        axes[-1].plot(xx, q0 + q1 * xx + q2 * xx ** 2, lw=2, color=COR_ANO.get(ano, '#2a78d6'), label=str(ano))
    axes[-1].set_title(f'{c} — curvas por ano (quadrática)', fontsize=10)
    axes[-1].set_xlabel('Consumo de gás (Nm³/h)')
    axes[-1].legend(fontsize=8, loc='upper left')
    axes[-1].grid(alpha=0.3)
    fig.suptitle(f'Caldeira {c}: geração de vapor × consumo de gás', fontweight='bold')
    plt.tight_layout()
    plt.show()

In [0]:
# Comparação entre anos: deixa evidentes as mudanças de eficiência
REF = None  # ano de referência (None = primeiro ano disponível de cada caldeira)
N_FAIXAS = 12  # faixas de carga para o consumo específico
for c, op in dados.items():
    anos_c = sorted(a for (cc, a) in curvas if cc == c)
    if not anos_c:
        continue
    ref = REF if REF in anos_c else anos_c[0]
    xx = np.linspace(max(curvas[(c, a)][5] for a in anos_c), min(curvas[(c, a)][6] for a in anos_c), 200)
    yq = {a: curvas[(c, a)][2] + curvas[(c, a)][3] * xx + curvas[(c, a)][4] * xx ** 2 for a in anos_c}
    faixas = np.linspace(op['vapor'].quantile(0.02), op['vapor'].quantile(0.98), N_FAIXAS + 1)
    centro = (faixas[:-1] + faixas[1:]) / 2
    fig, axes = plt.subplots(1, 3, figsize=(19, 5))
    for a in anos_c:
        cor = COR_ANO.get(a, '#888888')
        axes[0].plot(xx, yq[a], lw=2.2, color=cor, label=str(a))
        axes[1].plot(xx, 100 * (yq[a] - yq[ref]) / yq[ref], lw=2.2, color=cor, label=str(a))
        g = op[op['ano'] == a]
        e = (g['gas'] / g['vapor']).groupby(pd.cut(g['vapor'], faixas), observed=False)
        med, q25, q75, n = [np.array(s, dtype=float) for s in (e.median(), e.quantile(0.25), e.quantile(0.75), e.size())]
        ok = n >= 20
        axes[2].plot(centro[ok], med[ok], marker='o', ms=5, lw=2, color=cor, label=str(a))
        axes[2].fill_between(centro[ok], q25[ok], q75[ok], color=cor, alpha=0.12, linewidth=0)
    axes[1].axhline(0, color='#555555', lw=1)
    axes[0].set_title('Curva vapor × gás (faixa de gás comum a todos os anos)', fontsize=10)
    axes[1].set_title(f'Vapor gerado com o mesmo gás: diferença vs {ref} (%)', fontsize=10)
    axes[2].set_title('Consumo específico por faixa de carga (mediana e faixa 25–75%)', fontsize=10)
    axes[0].set_xlabel('Consumo de gás (Nm³/h)')
    axes[1].set_xlabel('Consumo de gás (Nm³/h)')
    axes[2].set_xlabel('Geração de vapor (t/h)')
    axes[0].set_ylabel('Geração de vapor (t/h)')
    axes[1].set_ylabel('Δ vapor (%)  — abaixo de 0 = pior que a referência')
    axes[2].set_ylabel('Gás por tonelada de vapor (Nm³/t)')
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, title='Ano')
    fig.suptitle(f'Caldeira {c}: mudanças na curva de consumo entre anos (referência {ref})', fontweight='bold')
    plt.tight_layout()
    plt.show()

# Resumo: consumo específico mediano por ano, com variação vs primeiro ano
fig, axes = plt.subplots(1, len(dados), figsize=(5.5 * len(dados), 4.4))
for ax, (c, op) in zip(np.atleast_1d(axes), dados.items()):
    e = (op['gas'] / op['vapor']).groupby(op['ano'])
    est = pd.DataFrame({'med': e.median(), 'q25': e.quantile(0.25), 'q75': e.quantile(0.75), 'n': e.size()})
    est = est[est['n'] >= MIN_HORAS_ANO]
    base = est['med'].iloc[0]
    ax.vlines(est.index, est['q25'], est['q75'], color='#cccccc', lw=8, zorder=0)
    ax.plot(est.index, est['med'], color='#444444', lw=1.5, zorder=1)
    ax.scatter(est.index, est['med'], s=80, color=[COR_ANO.get(a, '#888888') for a in est.index], edgecolors='white', linewidths=1.5, zorder=2)
    for a, r in est.iterrows():
        ax.annotate(f"{r['med']:.1f} ({100 * (r['med'] / base - 1):+.1f}%)", (a, r['med']), xytext=(0, 10), textcoords='offset points', ha='center', fontsize=8)
    ax.set_xticks(est.index)
    ax.margins(x=0.15, y=0.25)
    ax.set_title(f'Caldeira {c}', fontsize=10)
    ax.set_ylabel('Nm³ de gás por t de vapor (mediana)')
    ax.grid(alpha=0.3, axis='y')
fig.suptitle('Consumo específico por ano — barra cinza = faixa 25–75%; % = variação vs primeiro ano (↑ = mais gás por t)', fontweight='bold')
plt.tight_layout()
plt.show()

# Evolução mensal do consumo específico (linha tracejada = mediana do ano)
fig, axes = plt.subplots(len(dados), 1, figsize=(16, 3.2 * len(dados)), sharex=True)
for ax, (c, op) in zip(np.atleast_1d(axes), dados.items()):
    esp = op['gas'] / op['vapor']
    m = esp.resample('MS').median()
    m = m[op['vapor'].resample('MS').size() >= 100]
    ax.plot(m.index, m.values, color='#444444', lw=1.5, marker='o', ms=4)
    for a, g in esp.groupby(esp.index.year):
        ax.hlines(g.median(), g.index.min(), g.index.max(), colors=COR_ANO.get(a, '#888888'), linestyles='--', lw=2, label=f'{a}: {g.median():.1f}')
    ax.set_title(f'Caldeira {c} — consumo específico mensal (mediana, Nm³/t)', fontsize=10)
    ax.set_ylabel('Nm³/t')
    ax.legend(fontsize=8, ncol=4, loc='upper left')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
%pip install plotly

In [0]:
# Gráfico 3D por sistema: consumo de gás × pressão de vapor × vapor gerado (horas em operação) — gire com o mouse
import plotly.express as px
PRESSAO = {'208': '11PC208-01', '502': '11PC502-35', '550': '21PI550-15'}  # pressão de vapor (kg/cm²); na 550 há também 21PI550-95
AMOSTRA_POR_ANO = 3000  # pontos por ano em cada gráfico (mantém o gráfico leve)
resumo = []
for c, (tag_v, tag_g) in CALDEIRAS.items():
    df = pd.DataFrame({'vapor': carregar(tag_v), 'gas': carregar(tag_g)})
    df['pressao'] = carregar(PRESSAO[c]) if norm(PRESSAO[c]) in arquivos else np.nan
    op = df[df.index.isin(dados[c].index)].dropna().copy()
    if op.empty:
        print(f'Caldeira {c}: sem dados de pressão ({PRESSAO[c]}) nas horas em operação')
        continue
    op['ano'] = op.index.year.astype(str)
    amostra = pd.concat([g.sample(min(AMOSTRA_POR_ANO, len(g)), random_state=42) for _, g in op.groupby('ano')]).rename_axis('data_hora').reset_index()
    fig = px.scatter_3d(amostra, x='gas', y='pressao', z='vapor', color='ano', color_discrete_map={str(k): v for k, v in COR_ANO.items()}, category_orders={'ano': sorted(op['ano'].unique())}, opacity=0.6, hover_data=['data_hora'], labels={'gas': 'Consumo de gás (Nm³/h)', 'pressao': 'Pressão de vapor (kg/cm²)', 'vapor': 'Vapor gerado (t/h)', 'ano': 'Ano'}, title=f'Caldeira {c}: gás × pressão de vapor ({PRESSAO[c]}) × vapor gerado')
    fig.update_traces(marker=dict(size=2.5))
    fig.update_layout(height=650, legend=dict(itemsizing='constant'), margin=dict(l=0, r=0, t=50, b=0))
    fig.show()
    t = op.groupby('ano').agg(horas=('vapor', 'size'), vapor_medio=('vapor', 'mean'), gas_medio=('gas', 'mean'), pressao_media=('pressao', 'mean'), pressao_p5=('pressao', lambda s: s.quantile(0.05)), pressao_p95=('pressao', lambda s: s.quantile(0.95)))
    resumo.append(t.reset_index().assign(caldeira=c))
print('=== Médias anuais nas horas em operação ===')
display(pd.concat(resumo)[['caldeira', 'ano', 'horas', 'vapor_medio', 'gas_medio', 'pressao_media', 'pressao_p5', 'pressao_p95']].round(2))